# 03 — Benchmarking

## Objectifs pédagogiques

À la fin de ce notebook, vous saurez :
- utiliser `timeit` pour mesurer des micro-benchmarks fiables ;
- comprendre les pièges des micro-benchmarks (JIT, cache, GC) ;
- utiliser `pyperf` pour des benchmarks rigoureux et reproductibles ;
- comparer statistiquement deux implémentations ;
- appliquer une méthodologie de benchmark correcte.

## Prérequis — ce que vous connaissez déjà

Ce notebook s'adresse à un développeur Python **confirmé**. Vous maîtrisez déjà :
- le profiling CPU et mémoire (notebooks précédents) ;
- les structures de données et leur complexité algorithmique ;
- les décorateurs et context managers ;
- la ligne de commande Python.

## Plan

1. Pourquoi benchmarker correctement ?
2. `timeit` — micro-benchmarks
3. Pièges des micro-benchmarks
4. `timeit` en notebook : `%timeit` et `%%timeit`
5. `time.perf_counter` et `time.perf_counter_ns`
6. `pyperf` — benchmarks rigoureux
7. Comparer deux implémentations
8. Méthodologie de benchmark
9. Synthèse
10. Exercices
11. Ressources

---

## 1. Pourquoi benchmarker correctement ?

Le profiling identifie **où** le code est lent. Le benchmarking mesure **à quel point** il est lent (ou rapide), de manière **reproductible**.

| Erreur courante | Conséquence |
|---|---|
| `time.time()` avec un seul run | Bruit système masque le signal |
| Pas de warm-up | Cache CPU froid fausse la mesure |
| Comparer des moyennes sans écart-type | Impossible de conclure |
| Benchmarker en mode debug | Assertions et logging rajoutent du bruit |

---

## 2. `timeit` — micro-benchmarks

`timeit` fait partie de la bibliothèque standard. Il exécute un fragment de code un grand nombre de fois et retourne le temps total, en désactivant le garbage collector par défaut.

In [ ]:
import timeit

In [ ]:
# Mesurer la création d'une liste par compréhension
t = timeit.timeit("[x**2 for x in range(100)]", number=100_000)
print(f"100 000 itérations : {t:.3f} s")
print(f"Par itération : {t / 100_000 * 1e6:.1f} µs")

### `timeit.repeat()` — plusieurs séries

In [ ]:
temps = timeit.repeat("[x**2 for x in range(100)]", number=100_000, repeat=5)
print(f"Temps par série : {[f'{t:.3f}' for t in temps]}")
print(f"Minimum : {min(temps):.3f} s")

**Pourquoi prendre le minimum ?** Tout bruit système (GC, interruptions, swapping) ne peut que **ralentir**, jamais accélérer. Le minimum est donc la mesure la plus proche de la performance réelle.

### Avec un `setup`

In [ ]:
setup = "import math"
stmt = "math.sqrt(42)"
t = timeit.timeit(stmt, setup=setup, number=1_000_000)
print(f"math.sqrt(42) : {t / 1_000_000 * 1e9:.1f} ns/appel")

### Passer une callable

In [ ]:
def somme_boucle():
    total = 0
    for i in range(1000):
        total += i
    return total

t = timeit.timeit(somme_boucle, number=10_000)
print(f"somme_boucle : {t / 10_000 * 1e6:.1f} µs/appel")

---

## 3. Pièges des micro-benchmarks

### 3.1. Le GC peut interférer

In [ ]:
import gc

# Par défaut, timeit désactive le GC. Comparons :
stmt = "[list(range(100)) for _ in range(1000)]"

t_sans_gc = timeit.timeit(stmt, number=100)
t_avec_gc = timeit.timeit(stmt, setup="import gc; gc.enable()", number=100)
print(f"Sans GC : {t_sans_gc:.3f} s")
print(f"Avec GC : {t_avec_gc:.3f} s")

### 3.2. Le cache CPU change tout

In [ ]:
import random

# Accès séquentiel vs aléatoire
data = list(range(1_000_000))

def acces_sequentiel():
    return sum(data)

random_indices = random.sample(range(1_000_000), 1_000_000)

def acces_aleatoire():
    return sum(data[i] for i in random_indices)

t_seq = timeit.timeit(acces_sequentiel, number=10)
t_rng = timeit.timeit(acces_aleatoire, number=10)
print(f"Séquentiel : {t_seq:.3f} s")
print(f"Aléatoire  : {t_rng:.3f} s")
print(f"Ratio      : {t_rng / t_seq:.1f}x plus lent")

### 3.3. Ne pas benchmarker une constante

In [ ]:
# PIÈGE : le constant folding peut rendre le benchmark trivial
t1 = timeit.timeit("2 ** 100", number=1_000_000)
t2 = timeit.timeit("pow(2, 100)", number=1_000_000)
print(f"2 ** 100    : {t1 / 1_000_000 * 1e9:.1f} ns")
print(f"pow(2, 100) : {t2 / 1_000_000 * 1e9:.1f} ns")
# 2 ** 100 est plié par le compilateur : il charge une constante !

---

## 4. `timeit` en notebook : `%timeit` et `%%timeit`

IPython (et donc Jupyter) fournit des magics `%timeit` (une ligne) et `%%timeit` (une cellule entière). Ils choisissent automatiquement le nombre d'itérations et affichent des statistiques.

In [ ]:
%timeit sum(range(10_000))

In [ ]:
%timeit -n 1000 -r 7 sorted(list(range(1000)))

Options utiles :
- `-n` : nombre d'exécutions par boucle
- `-r` : nombre de répétitions (séries)
- `-o` : retourne un objet `TimeitResult` pour un traitement ultérieur

In [ ]:
result = %timeit -o sum(range(10_000))
print(f"Best : {result.best * 1e6:.1f} µs")
print(f"Worst: {result.worst * 1e6:.1f} µs")
print(f"Mean : {result.average * 1e6:.1f} µs")

### `%%timeit` — cellule entière

In [ ]:
%%timeit
data = list(range(10_000))
sorted_data = sorted(data, reverse=True)

---

## 5. `time.perf_counter` et `time.perf_counter_ns`

Pour des mesures ponctuelles dans du code de production, `time.perf_counter()` (ou sa version nanoseconde) est le chronomètre le plus précis.

In [ ]:
import time

In [ ]:
start = time.perf_counter()
total = sum(range(1_000_000))
elapsed = time.perf_counter() - start
print(f"Temps : {elapsed * 1000:.2f} ms")

In [ ]:
# Version nanoseconde pour les mesures très courtes
start_ns = time.perf_counter_ns()
x = 42 ** 100
elapsed_ns = time.perf_counter_ns() - start_ns
print(f"Temps : {elapsed_ns} ns")

### Ne pas utiliser `time.time()` pour le benchmarking

| Horloge | Résolution | Monotone | Usage |
|---|---|---|---|
| `time.time()` | ~1 µs | Non (NTP) | Timestamps absolus |
| `time.perf_counter()` | ~100 ns | Oui | Benchmarking |
| `time.monotonic()` | ~1 µs | Oui | Timeouts |
| `time.process_time()` | ~1 µs | Oui | Temps CPU pur (hors sleep) |

### Créer un context manager de chronométrage

In [ ]:
from contextlib import contextmanager

@contextmanager
def chrono(label=""):
    t0 = time.perf_counter_ns()
    yield
    dt = time.perf_counter_ns() - t0
    if dt < 1000:
        print(f"{label}: {dt} ns")
    elif dt < 1_000_000:
        print(f"{label}: {dt / 1000:.1f} µs")
    elif dt < 1_000_000_000:
        print(f"{label}: {dt / 1_000_000:.2f} ms")
    else:
        print(f"{label}: {dt / 1_000_000_000:.3f} s")

In [ ]:
with chrono("sum(range(1M))"):
    total = sum(range(1_000_000))

---

## 6. `pyperf` — benchmarks rigoureux

`pyperf` est la référence pour des benchmarks **statistiquement significatifs**. Il gère automatiquement :
- le warm-up (calibration) ;
- l'isolation des processus (fork) ;
- le calcul de la moyenne, médiane, écart-type ;
- la détection de résultats instables.

> **Installation :** `pip install pyperf`

### Utilisation en ligne de commande

```bash
# Benchmark simple
python -m pyperf timeit "sum(range(10_000))"

# Avec setup
python -m pyperf timeit -s "data = list(range(10_000))" "sorted(data)"

# Sauvegarder les résultats
python -m pyperf timeit -o result.json "sum(range(10_000))"

# Comparer deux résultats
python -m pyperf compare_to baseline.json optimized.json
```

### Utilisation programmatique

In [ ]:
# Note : pyperf crée des sous-processus, donc il ne fonctionne pas
# directement dans un notebook Jupyter. On montre l'API ici,
# et on l'exécute en script.

print("""
# benchmark_sum.py
import pyperf

runner = pyperf.Runner()
runner.timeit(
    name="sum(range(10_000))",
    stmt="sum(range(10_000))",
)
""")

### Interpréter les résultats de `pyperf`

```
sum(range(10_000)): Mean +- std dev: 45.2 us +- 0.8 us
```

| Métrique | Signification |
|---|---|
| Mean | Moyenne arithmétique |
| std dev | Écart-type (si > 10% de la moyenne → résultat instable) |
| Median | Valeur médiane (plus robuste aux outliers) |
| Min/Max | Bornes extrêmes |

### `pyperf compare_to` — comparaison statistique

```
$ python -m pyperf compare_to baseline.json optimized.json
Mean +- std dev: [baseline] 45.2 us +- 0.8 us
                 [optimized] 32.1 us +- 0.5 us
Benchmark: 1.41x faster (-29%)
Significant (t-test)
```

`pyperf` effectue un **t-test de Student** pour déterminer si la différence est statistiquement significative. Si ce n'est pas le cas, il affiche « Not significant ».

---

## 7. Comparer deux implémentations

In [ ]:
def concat_plus(n):
    s = ""
    for i in range(n):
        s += str(i)
    return s

def concat_join(n):
    return "".join(str(i) for i in range(n))

In [ ]:
for n in [100, 1_000, 10_000]:
    t_plus = min(timeit.repeat(lambda: concat_plus(n), number=100, repeat=3))
    t_join = min(timeit.repeat(lambda: concat_join(n), number=100, repeat=3))
    ratio = t_plus / t_join
    print(f"n={n:>6d}  +=: {t_plus:.4f}s  join: {t_join:.4f}s  ratio: {ratio:.1f}x")

La concaténation par `+=` est O(n^2) en théorie (chaque `+=` crée une nouvelle chaîne), alors que `join` est O(n). La différence se voit clairement quand n augmente.

### Comparer list vs deque pour appendleft

In [ ]:
from collections import deque

def insert_list(n):
    lst = []
    for i in range(n):
        lst.insert(0, i)
    return lst

def append_deque(n):
    d = deque()
    for i in range(n):
        d.appendleft(i)
    return d

for n in [1_000, 10_000]:
    t_list = min(timeit.repeat(lambda: insert_list(n), number=10, repeat=3))
    t_deque = min(timeit.repeat(lambda: append_deque(n), number=10, repeat=3))
    print(f"n={n:>6d}  list.insert(0): {t_list:.4f}s  deque.appendleft: {t_deque:.4f}s  ratio: {t_list/t_deque:.0f}x")

### Comparer `in` sur list, set, dict

In [ ]:
N = 100_000
lst = list(range(N))
st = set(range(N))
dt = dict.fromkeys(range(N))

# Chercher le dernier élément (pire cas pour list)
cible = N - 1

t_list = timeit.timeit(lambda: cible in lst, number=100)
t_set = timeit.timeit(lambda: cible in st, number=100)
t_dict = timeit.timeit(lambda: cible in dt, number=100)

print(f"list : {t_list:.4f}s")
print(f"set  : {t_set:.6f}s")
print(f"dict : {t_dict:.6f}s")
print(f"set vs list : {t_list / t_set:.0f}x plus rapide")

---

## 8. Méthodologie de benchmark

1. **Isolez** ce que vous mesurez : séparez le setup du code mesuré.
2. **Répétez** : au moins 3-5 séries, prenez le **minimum**.
3. **Warm-up** : faites quelques runs non comptés pour chauffer le cache CPU.
4. **Désactivez le bruit** : fermez les autres applications, désactivez le turbo boost si possible.
5. **Variez les entrées** : testez avec plusieurs tailles pour voir la tendance.
6. **Comparez statistiquement** : utilisez `pyperf compare_to` ou un t-test.
7. **Documentez** : notez la machine, la version Python, les paramètres.

### Checklist avant benchmark

```
[ ] Code de production (pas de mode debug)
[ ] GC désactivé ou contrôlé
[ ] Données réalistes
[ ] Plusieurs tailles testées
[ ] Minimum de 3 répétitions
[ ] Machine au repos
[ ] Version Python documentée
[ ] Résultats avec écart-type
```

---

## 9. Synthèse

| Outil | Usage | Fiabilité |
|---|---|---|
| `timeit.timeit()` | Micro-benchmarks rapides | Bonne |
| `%timeit` / `%%timeit` | Benchmarks en notebook | Bonne (pratique) |
| `time.perf_counter()` | Chronométrage ponctuel | Moyenne (un seul run) |
| `pyperf` | Benchmarks rigoureux, comparaisons | Excellente |

**Règles à retenir :**
- Prenez le **minimum** de plusieurs séries, pas la moyenne.
- `timeit` désactive le GC par défaut — c'est voulu.
- Le constant folding peut rendre un benchmark trivial — vérifiez avec `dis`.
- `pyperf` est le gold standard pour publier des résultats.
- Variez les tailles d'entrée pour voir la **tendance** (O(n) vs O(n^2)).

---

## 10. Exercices

### Exercice 1 — Comparer trois façons de créer une liste *(facile)*

Utilisez `timeit` pour comparer :
1. `list(range(10_000))`
2. `[x for x in range(10_000)]`
3. `[*range(10_000)]`

Affichez le temps par appel en microsecondes.

In [ ]:
# Votre code ici


In [ ]:
# ▶ Une fois votre solution écrite ci-dessus, exécutez cette cellule
# pour signaler à votre formateur que vous avez tenté l'exercice.
import sys
from pathlib import Path
for _p in (Path.cwd(), *Path.cwd().parents):
    if (_p / "_common" / "utils_pedagogie.py").exists():
        sys.path.insert(0, str(_p / "_common")); break
from utils_pedagogie import marquer_tentative
marquer_tentative(notebook="03_Benchmarking", exercice=1)


<details>
<summary>📖 Voir la correction</summary>

```python
import timeit

approches = {
    "list(range)": "list(range(10_000))",
    "comprehension": "[x for x in range(10_000)]",
    "unpack": "[*range(10_000)]",
}

for nom, stmt in approches.items():
    t = min(timeit.repeat(stmt, number=1000, repeat=5))
    print(f"{nom:20s} : {t / 1000 * 1e6:.1f} µs/appel")
```

</details>

### Exercice 2 — Benchmark paramétrique *(moyen)*

Écrire une fonction `benchmark_scalabilite(fn, tailles, number=100)` qui :
1. Pour chaque taille dans `tailles`, appelle `fn(taille)` via `timeit` ;
2. Affiche un tableau avec la taille, le temps, et le ratio temps/taille.

Testez avec `sorted(list(range(n)))` pour n dans [1000, 5000, 10000, 50000].

In [ ]:
# Votre code ici


In [ ]:
# ▶ Une fois votre solution écrite ci-dessus, exécutez cette cellule
# pour signaler à votre formateur que vous avez tenté l'exercice.
import sys
from pathlib import Path
for _p in (Path.cwd(), *Path.cwd().parents):
    if (_p / "_common" / "utils_pedagogie.py").exists():
        sys.path.insert(0, str(_p / "_common")); break
from utils_pedagogie import marquer_tentative
marquer_tentative(notebook="03_Benchmarking", exercice=2)


<details>
<summary>📖 Voir la correction</summary>

```python
import timeit

def benchmark_scalabilite(fn, tailles, number=100):
    print(f"{'Taille':>10s} {'Temps (ms)':>12s} {'Temps/n (ns)':>14s}")
    print("-" * 40)
    for n in tailles:
        t = min(timeit.repeat(lambda: fn(n), number=number, repeat=3))
        temps_ms = t / number * 1000
        temps_par_n = t / number / n * 1e9
        print(f"{n:>10d} {temps_ms:>12.3f} {temps_par_n:>14.1f}")

def tri_liste(n):
    return sorted(list(range(n)))

benchmark_scalabilite(tri_liste, [1_000, 5_000, 10_000, 50_000])
# Le ratio temps/n devrait être ~ constant (O(n log n) / n = O(log n))
```

</details>

### Exercice 3 — Décorateur `@benchmark` *(moyen)*

Écrire un décorateur `@benchmark(repeat=5)` qui mesure chaque appel de la fonction décorée et stocke les résultats dans un attribut `.timings`.

```python
@benchmark(repeat=5)
def ma_fonction(n):
    return sum(range(n))

ma_fonction(100_000)
ma_fonction(200_000)
print(ma_fonction.timings)
```

In [ ]:
# Votre code ici


In [ ]:
# ▶ Une fois votre solution écrite ci-dessus, exécutez cette cellule
# pour signaler à votre formateur que vous avez tenté l'exercice.
import sys
from pathlib import Path
for _p in (Path.cwd(), *Path.cwd().parents):
    if (_p / "_common" / "utils_pedagogie.py").exists():
        sys.path.insert(0, str(_p / "_common")); break
from utils_pedagogie import marquer_tentative
marquer_tentative(notebook="03_Benchmarking", exercice=3)


<details>
<summary>📖 Voir la correction</summary>

```python
import time
import functools

def benchmark(repeat=5):
    def decorator(fn):
        @functools.wraps(fn)
        def wrapper(*args, **kwargs):
            times = []
            result = None
            for _ in range(repeat):
                t0 = time.perf_counter_ns()
                result = fn(*args, **kwargs)
                times.append(time.perf_counter_ns() - t0)
            best = min(times)
            wrapper.timings.append({
                "args": args, "kwargs": kwargs,
                "best_ns": best, "all_ns": times,
            })
            return result
        wrapper.timings = []
        return wrapper
    return decorator

@benchmark(repeat=5)
def ma_fonction(n):
    return sum(range(n))

ma_fonction(100_000)
ma_fonction(200_000)
for t in ma_fonction.timings:
    print(f"args={t['args']}  best={t['best_ns']/1e6:.2f} ms")
```

</details>

### Exercice 4 — Trouver le point de bascule *(difficile)*

Deux algorithmes de recherche : linéaire O(n) et binaire O(log n). La recherche binaire a un overhead plus élevé par appel. Trouvez le **point de bascule** (la taille n à partir de laquelle la recherche binaire devient plus rapide).

```python
def recherche_lineaire(lst, cible):
    for x in lst:
        if x == cible:
            return True
    return False

import bisect
def recherche_binaire(lst, cible):
    i = bisect.bisect_left(lst, cible)
    return i < len(lst) and lst[i] == cible
```

In [ ]:
# Votre code ici


In [ ]:
# ▶ Une fois votre solution écrite ci-dessus, exécutez cette cellule
# pour signaler à votre formateur que vous avez tenté l'exercice.
import sys
from pathlib import Path
for _p in (Path.cwd(), *Path.cwd().parents):
    if (_p / "_common" / "utils_pedagogie.py").exists():
        sys.path.insert(0, str(_p / "_common")); break
from utils_pedagogie import marquer_tentative
marquer_tentative(notebook="03_Benchmarking", exercice=4)


<details>
<summary>📖 Voir la correction</summary>

```python
import timeit
import bisect

def recherche_lineaire(lst, cible):
    for x in lst:
        if x == cible:
            return True
    return False

def recherche_binaire(lst, cible):
    i = bisect.bisect_left(lst, cible)
    return i < len(lst) and lst[i] == cible

for n in [10, 50, 100, 500, 1000, 5000, 10000]:
    data = list(range(n))
    cible = n - 1  # pire cas pour linéaire
    t_lin = min(timeit.repeat(lambda: recherche_lineaire(data, cible), number=1000, repeat=3))
    t_bin = min(timeit.repeat(lambda: recherche_binaire(data, cible), number=1000, repeat=3))
    gagnant = "binaire" if t_bin < t_lin else "linéaire"
    print(f"n={n:>6d}  linéaire: {t_lin:.5f}s  binaire: {t_bin:.5f}s  → {gagnant}")
```

</details>

---

## 11. Ressources

- [Module `timeit` — documentation officielle](https://docs.python.org/3/library/timeit.html)
- [`pyperf` — documentation](https://pyperf.readthedocs.io/)
- [IPython `%timeit` magic](https://ipython.readthedocs.io/en/stable/interactive/magics.html#magic-timeit)
- [Benchmarking Python — Real Python](https://realpython.com/python-timer/)